# Named Entity Recognition — Train the spaCy Model

This notebook converts CoNLL-2003 token-level IOB annotations into character offsets and trains a custom spaCy NER model.

The project deliberately focuses on:
- PERSON
- LOCATION
- ORGANIZATION

In [1]:
import random
from pathlib import Path

import spacy
from datasets import load_dataset
from spacy.training import Example
from spacy.util import minibatch, compounding

random.seed(42)

dataset = load_dataset(
    "lhoestq/conll2003",
    trust_remote_code=True
)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'lhoestq/conll2003' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


In [2]:
LABEL_MAP = {
    0: "O",
    1: "B-PER",
    2: "I-PER",
    3: "B-ORG",
    4: "I-ORG",
    5: "B-LOC",
    6: "I-LOC",
    7: "B-MISC",
    8: "I-MISC",
}

KEEP = {"PER": "PERSON", "ORG": "ORG", "LOC": "LOC"}

def iob_to_spans(tokens, tag_ids):
    spans = []
    current_label = None
    start_token = None

    for i, tag_id in enumerate(tag_ids + [0]):
        tag = LABEL_MAP[tag_id]

        if tag.startswith("B-"):
            if current_label is not None:
                spans.append((start_token, i, current_label))
            current_label = KEEP.get(tag[2:])
            start_token = i if current_label else None

        elif tag.startswith("I-"):
            # Continue only if it belongs to a label used in the project.
            continue

        else:
            if current_label is not None:
                spans.append((start_token, i, current_label))
            current_label = None
            start_token = None

    return spans

In [3]:
def build_examples(rows, limit=None):
    examples = []

    for row in rows[:limit] if limit else rows:
        tokens = row["tokens"]
        tags = row["ner_tags"]

        # CoNLL words are joined with spaces for this demonstration.
        text = " ".join(tokens)
        offsets = []
        cursor = 0

        for token in tokens:
            start = text.find(token, cursor)
            end = start + len(token)
            offsets.append((start, end))
            cursor = end + 1

        entities = []
        for start_token, end_token, label in iob_to_spans(tokens, tags):
            start_char = offsets[start_token][0]
            end_char = offsets[end_token - 1][1]
            entities.append((start_char, end_char, label))

        examples.append((text, {"entities": entities}))

    return examples

# A moderate subset keeps training practical on a normal laptop.
def build_examples(rows, limit=None):
    examples = []

    if limit is not None:
        rows = rows.select(range(min(limit, len(rows))))

    for row in rows:
        tokens = row["tokens"]
        tags = row["ner_tags"]

        text = " ".join(tokens)
        offsets = []
        cursor = 0

        for token in tokens:
            start = text.find(token, cursor)
            end = start + len(token)
            offsets.append((start, end))
            cursor = end + 1

        entities = []
        for start_token, end_token, label in iob_to_spans(tokens, tags):
            entities.append((
                offsets[start_token][0],
                offsets[end_token - 1][1],
                label,
            ))

        examples.append((text, {"entities": entities}))

    return examples

train_data = build_examples(dataset["train"], limit=12000)
valid_data = build_examples(dataset["validation"], limit=2500)

len(train_data), len(valid_data)

(12000, 2500)

In [ ]:
nlp = spacy.blank("en")
ner = nlp.add_pipe("ner")

for label in KEEP.values():
    ner.add_label(label)

optimizer = nlp.begin_training()

for epoch in range(10):
    # Rebuild the training data if this cell is run before the data-preparation cell.
    if "train_data" not in globals():
        train_data = build_examples(dataset["train"], limit=12000)

    random.shuffle(train_data)
    losses = {}

    batches = minibatch(train_data, size=compounding(4.0, 32.0, 1.001))

    for batch in batches:
        examples = []
        for text, annotations in batch:
            doc = nlp.make_doc(text)
            examples.append(Example.from_dict(doc, annotations))

        nlp.update(
            examples,
            drop=0.25,
            sgd=optimizer,
            losses=losses,
        )

    print(f"Epoch {epoch + 1:02d} | Loss: {losses.get('ner', 0):.2f}")

Epoch 01 | Loss: 11856.67


In [ ]:
model_path = Path("../ner_model")
nlp.to_disk(model_path)

print(f"Saved trained model to: {model_path.resolve()}")

Saved trained model to: C:\Users\user\Desktop\Named-Entity-Recognition-NER\notebooks\ner_model


In [ ]:
# Quick sanity check
test_text = "Barack Obama visited London and later spoke with Microsoft executives."

doc = nlp(test_text)

for ent in doc.ents:
    print(f"{ent.text:20} -> {ent.label_}")

Barack Obama         -> PERSON
London               -> LOC
Microsoft            -> ORG


## Notes for an Interview

The important implementation detail is the conversion from token-level IOB tags to character-level spans.

spaCy training examples need entity offsets such as:

`(0, 12, "PERSON")`

rather than only `B-PER` / `I-PER` labels. This notebook bridges those two representations.